In [ ]:
pip install numbers-parser

In [4]:
from numbers_parser import Documentimport csv

In [14]:
# Original: converted .numbers to CSV (step skipped — starting from preprocessed data)


In [15]:
sheet = doc.sheets[0]table = sheet.tables[0]

In [16]:
with open(r"../data/subjects.csv", 'w', newline='', encoding='utf-8') as f:    writer = csv.writer(f)    for r in range(table.num_rows):        row =  [table.cell(r, c).value for c in range(table.num_cols)]        writer.writerow(row)

In [17]:
import pandas as pd

In [23]:
df = pd.read_csv(r"../data/subjects.csv")

In [26]:
df

In [27]:
df.drop(columns=['Unnamed: 64'], inplace=True)

In [29]:
df = df.dropna(axis=1, how='all')

In [33]:
df = df.drop(index=range(22, 28)).reset_index(drop=True)

In [38]:
df.isnull().sum(axis=1).sort_values(ascending=False)

In [42]:
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [45]:
df = df.drop(columns=['H49_other_relationship_satisfaction','C15_social_status'])

In [46]:
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [47]:
exclude = ['name_or_code', 'id', 'A2_meeting_route',           'y1_attracted', 'y2_attraction_intensity', 'y3_dated',            'y4_relationship_satisfaction']fill_cols = [c for c in df.columns if c not in exclude]df[fill_cols] = df[fill_cols].fillna(df[fill_cols].median())

In [48]:
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [51]:
# 수치형 컬럼만 상관계수 확인 (타깃: y2_attraction_intensity 기준)corr = df.select_dtypes(include='number').corr()corr[['y2_attraction_intensity']].sort_values(by='y2_attraction_intensity', ascending=False)

In [52]:
df['A1_age_diff_abs'] = df['A1_age_diff'].abs()

In [53]:
import matplotlib.pyplot as pltimport seaborn as snstarget_corr = corr[['y2_attraction_intensity']].drop(['y1_attracted','y2_attraction_intensity','y3_dated','y4_relationship_satisfaction','id','year']).sort_values(by='y2_attraction_intensity', ascending=False)plt.figure(figsize=(8, 12))sns.barplot(x='y2_attraction_intensity', y=target_corr.index, data=target_corr)plt.title('Correlation with Attraction Intensity')plt.tight_layout()plt.show()

In [54]:
top10 = ['G45_conversation_fun', 'H50_access_frequency', 'E34_push_pull_tension',         'B9_scent_sensory', 'B7_voice', 'D20_cognitive_sharpness',         'D21_humor_match', 'I53_thought_sync_frequency',          'F36_value_similarity', 'D24_clumsy_but_trying']sns.heatmap(df[top10].corr(), annot=True, cmap='RdPu', fmt='.2f')plt.tight_layout()plt.show()

In [56]:
pip install statsmodels

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor# 수치형 피처만X = df[fill_cols].select_dtypes(include='number')vif = pd.DataFrame({    'feature': X.columns,    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]}).sort_values('VIF', ascending=False)print(vif.head(15))

In [60]:
# y2와 상관 0.5 이상인 피처 뽑기target_corr = corr['y2_attraction_intensity'].drop(['y1_attracted','y2_attraction_intensity','y3_dated','y4_relationship_satisfaction','id','year'])strong = target_corr[target_corr.abs() >= 0.5].sort_values(ascending=False)print(strong)print(f'\n총 {len(strong)}개')

In [ ]:
strong_cols = strong.index.tolist()strong_corr = df[strong_cols].corr().abs()upper = strong_corr.where(np.triu(np.ones(strong_corr.shape), k=1).astype(bool))pairs = [(upper.columns[j], upper.index[i], upper.iloc[i,j])                                    ]

In [63]:
import numpy as np

In [64]:
strong_cols = strong.index.tolist()strong_corr = df[strong_cols].corr().abs()upper = strong_corr.where(np.triu(np.ones(strong_corr.shape), k=1).astype(bool))pairs = [(upper.columns[j], upper.index[i], upper.iloc[i,j])          for i in range(len(upper)) for j in range(len(upper.columns))          if upper.iloc[i,j] > 0.7]for a, b, val in sorted(pairs, key=lambda x: -x[2]):    print(f'{a}  ↔  {b}: {val:.2f}')

In [65]:
# 0.8 이상 짝에서 y2 상관 낮은 쪽 자동 droptarget_corr = corr['y2_attraction_intensity']to_drop = set()for a, b, val in pairs:    if val >= 0.8:        if abs(target_corr[a]) >= abs(target_corr[b]):            to_drop.add(b)        else:            to_drop.add(a)print('Drop:', to_drop)print('남은 피처:', [c for c in strong.index if c not in to_drop])

In [67]:
to_drop

In [68]:
selected_features = [c for c in strong.index if c not in to_drop]print(selected_features)print(f'최종 피처: {len(selected_features)}개')

In [69]:
from sklearn.tree import DecisionTreeRegressorX = df[selected_features]y = df['y2_attraction_intensity']model = DecisionTreeRegressor(random_state=42)

In [70]:
model.fit(X, y)

In [71]:
importance = pd.DataFrame({    'feature': selected_features,    'importance': model.feature_importances_}).sort_values('importance', ascending=False)

In [72]:
print(importance)

In [73]:
from sklearn.ensemble import RandomForestRegressormodel_rf = RandomForestRegressor(n_estimators=100, random_state=42)model_rf.fit(X, y)importance_rf = pd.DataFrame({    'feature':selected_features,    'importance': model_rf.feature_importances_}).sort_values('importance', ascending=False)print(importance_rf)

In [77]:
from sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LinearRegressionX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)model = LinearRegression()model.fit(X_train, y_train)pred = model.predict(X_test)print(pd.DataFrame({'실제': y_test.values, '예측': pred}))

In [78]:
print(pd.DataFrame({    '이름': df.loc[y_test.index, 'name_or_code'].values,    '실제': y_test.values,     '예측': pred}))

In [80]:
for rs in [42, 10, 7, 99, 123]:    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs)    model = LinearRegression()    model.fit(X_train, y_train)    pred = model.predict(X_test)        result = pd.DataFrame({        '이름': df.loc[y_test.index, 'name_or_code'].values,        '실제': y_test.values,        '예측': pred.round(1)    })    print(f'\n--- random_state={rs} ---')    print(result)

In [81]:
from sklearn.metrics import mean_squared_error, r2_scoreprint(f'MSE: {mean_squared_error(y_test, pred):.2f}')print(f'R²: {r2_score(y_test, pred):.2f}')

In [82]:
result = pd.DataFrame({    '이름': df.loc[y_test.index, 'name_or_code'].values,    '실제': y_test.values,    '예측': pred.round(1),    '차이': (y_test.values - pred).round(1)}).sort_values('차이')print(result)

In [87]:
for rs in [17, 794, 96, 189, 81]:    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs)    model = LinearRegression()    model.fit(X_train, y_train)    pred = model.predict(X_test)        result = pd.DataFrame({        '이름': df.loc[y_test.index, 'name_or_code'].values,        '실제': y_test.values,        '예측': pred.round(1),        '차이': (y_test.values - pred).round(1)    }).sort_values('차이')        print(f'\n--- random_state={rs} ---')    print(result)

In [ ]:
names = ['Subject #02', 'Subject #20']check = df[df['name_or_code'].isin(names)]print(check[top5 + ['y2_attraction_intensity', 'D17_honesty_transparency', 'E30_emotional_safety']].T)

In [89]:
from sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeans# 피처 준비 (top5 말고 좀 넓게 써도 돼 — 군집은 다양한 차원에서 그룹을 찾으니까)X_cluster = df[selected_features]  # 12개 피처# 거리 기반이라 스케일링 필수 (06 와인에서 강조했던 거)scaler = StandardScaler()X_scaled = scaler.fit_transform(X_cluster)# 엘보우로 최적 k 찾기inertia = []for k in range(1, 8):    km = KMeans(n_clusters=k, random_state=42)    km.fit(X_scaled)    inertia.append(km.inertia_)plt.plot(range(1, 8), inertia, marker='o')plt.xlabel('k')plt.ylabel('Inertia')plt.title('Elbow Method')plt.show()

In [96]:
X_cluster

In [92]:
from sklearn.decomposition import PCAkm = KMeans(n_clusters=3, random_state=42)km.fit(X_scaled)df['cluster'] = km.labels_# PCA로 2D 시각화 (07 야구 노트북에서 했던 거)pca = PCA(n_components=2)pca_result = pca.fit_transform(X_scaled)plt.rcParams['font.family'] = 'Malgun Gothic'plt.rcParams['axes.unicode_minus'] = Falseplt.figure(figsize=(10, 6))for c in range(3):    mask = df['cluster'] == c    plt.scatter(pca_result[mask, 0], pca_result[mask, 1],                 label=f'Cluster {c}', s=100)    # 이름 표시    for idx in df[mask].index:        plt.annotate(df.loc[idx, 'name_or_code'],                      (pca_result[idx, 0], pca_result[idx, 1]), fontsize=8)plt.xlabel('PC1')plt.ylabel('PC2')plt.title('Who did you date? (3 Clusters)')plt.legend()plt.tight_layout()plt.show()

In [93]:
for c in range(3):    names = df[df['cluster'] == c]['name_or_code'].tolist()    mean_y2 = df[df['cluster'] == c]['y2_attraction_intensity'].mean()    print(f'Cluster {c}: {names}')    print(f'  평균 호감도: {mean_y2:.1f}\n')

In [99]:
km5 = KMeans(n_clusters=8, random_state=42)km5.fit(X_scaled)df['cluster'] = km5.labels_pca_result = pca.transform(X_scaled)plt.figure(figsize=(10, 6))for c in range(8):    mask = df['cluster'] == c    plt.scatter(pca_result[mask, 0], pca_result[mask, 1],                 label=f'Cluster {c}', s=100)    for idx in df[mask].index:        plt.annotate(df.loc[idx, 'name_or_code'],                      (pca_result[idx, 0], pca_result[idx, 1]), fontsize=8)plt.xlabel('PC1')plt.ylabel('PC2')plt.title('Who did you date? (5 Clusters)')plt.legend()plt.tight_layout()plt.show()

In [101]:
from sklearn.metrics import silhouette_scorefor k in [3, 5, 8]:    km = KMeans(n_clusters=k, random_state=42)    labels = km.fit_predict(X_scaled)    score = silhouette_score(X_scaled, labels)    print(f'k={k}: 실루엣 = {score:.3f}')

In [103]:
km3 = KMeans(n_clusters=3, random_state=42)df['cluster'] = km3.fit_predict(X_scaled)for c in range(3):    members = df[df['cluster'] == c]['name_or_code'].tolist()    mean_y2 = df[df['cluster'] == c]['y2_attraction_intensity'].mean()    print(f'Cluster {c}: {members}')    print(f'  평균 호감도: {mean_y2:.1f}\n')

In [104]:
from sklearn.tree import DecisionTreeClassifier, plot_treeX_cls = df[selected_features]  # 12개 피처y_cls = df['y3_dated']tree = DecisionTreeClassifier(max_depth=3, random_state=42)tree.fit(X_cls, y_cls)plt.figure(figsize=(20, 10))plot_tree(tree, feature_names=selected_features,           class_names=['안사귐', '사귐'], filled=True, fontsize=10)plt.title('연애 의사결정 트리')plt.tight_layout()plt.show()

In [108]:
from sklearn.ensemble import IsolationForestiso = IsolationForest(contamination=0.15, random_state=7)df['outlier'] = iso.fit_predict(X_scaled)# -1이 이상치, 1이 정상outliers = df[df['outlier'] == -1][['name_or_code', 'y2_attraction_intensity', 'y3_dated']]normals = df[df['outlier'] == 1][['name_or_code', 'y2_attraction_intensity', 'y3_dated']]print('=== 네 패턴에서 벗어나는 사람 ===')print(outliers)print(f'\n=== 나머지 {len(normals)}명 ===')print(normals)

In [112]:
partner = df[df['name_or_code'] == 'Subject #01']avg = df[selected_features].mean()comp = pd.DataFrame({    'Subject #01': partner[selected_features].values[0],    '평균': avg.values}, index=selected_features)comp['차이'] = comp['Subject #01'] - comp['평균']print(comp.sort_values('차이'))

In [113]:
pip install shap

In [114]:
import numpy as np# subject_01 vs 평균features = selected_featuressubject_01 = df[df['name_or_code'] == 'Subject #01'][features].values[0]avg = df[features].mean().values# 레이더 차트angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()angles += angles[:1]fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))ax.plot(angles, list(subject_01) + [subject_01[0]], 'o-', label='Subject #01', linewidth=2)ax.plot(angles, list(avg) + [avg[0]], 'o--', label='평균', linewidth=2)ax.fill(angles, list(subject_01) + [subject_01[0]], alpha=0.15)ax.set_xticks(angles[:-1])ax.set_xticklabels(features, fontsize=8)ax.set_ylim(0, 10)ax.legend(loc='upper right', fontsize=12)ax.set_title('subject_01 vs 평균 프로필', fontsize=14)plt.tight_layout()plt.show()

In [117]:
# 피처별 기여도 (permutation importance로 대체)from sklearn.inspection import permutation_importancemodel_rf = RandomForestRegressor(n_estimators=100, random_state=42)model_rf.fit(df[selected_features], df['y2_attraction_intensity'])# subject_01의 예측 분해idx = df[df['name_or_code'] == 'Subject #01'].index[0]subject_01_vals = df.loc[idx, selected_features]avg_vals = df[selected_features].mean()# 피처별 편차 × importance = 기여도 근사imp = model_rf.feature_importances_contrib = (subject_01_vals.values - avg_vals.values) * impresult = pd.DataFrame({    'feature': selected_features,    '기여도': contrib}).sort_values('기여도', ascending=True)plt.figure(figsize=(10, 6))colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in result['기여도']]plt.barh(result['feature'], result['기여도'], color=colors)plt.title('subject_01한테 끌리는 이유 분해')plt.xlabel('끌림 기여도 (+ 올림 / - 내림)')plt.tight_layout()plt.show()

In [118]:
from sklearn.metrics.pairwise import euclidean_distances# subject_01 vs 나머지 거리 계산idx = df[df['name_or_code'] == 'Subject #01'].index[0]distances = euclidean_distances(X_scaled[idx].reshape(1, -1), X_scaled)[0]df['subject_01와의_거리'] = distancesneighbors = df[df['name_or_code'] != 'Subject #01'][['name_or_code', 'subject_01와의_거리', 'y2_attraction_intensity']].sort_values('subject_01와의_거리')print(neighbors.head(10))